# dim_video_scd1 (SCD 1)

One row per `video_id`, latest values + YouTube category name + LLM video type (`stg_video_classification`). `video_key = xxhash64(video_id)`.

Run by `dims_scd1` (task `gold_dims_scd1`); widgets `load_type` (incremental / full), `from_date`, `to_date` (Vietnam days, empty = yesterday .. tomorrow). The live dim is unchanged.

In [ ]:
%run ./_scd

# Set up environment

In [ ]:
for layer in ("silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"])
dbutils.widgets.text("from_date", "")   # yyyy-MM-dd, Vietnam day, inclusive; empty = yesterday
dbutils.widgets.text("to_date", "")     # exclusive; empty = tomorrow

silver = dbutils.widgets.get("silver")
gold = dbutils.widgets.get("gold")
load_type = dbutils.widgets.get("load_type")
from_date, to_date = get_incremental_dates(dbutils.widgets.get("from_date"), dbutils.widgets.get("to_date"))
print(f"load_type = {load_type}, window = [{from_date}, {to_date})")

# Read Source Table

In [ ]:
stg_videos = spark.table(f"{silver}.stg_videos")
stg_videos_latest = spark.table(f"{silver}.stg_videos_latest")
classification_table = f"{silver}.stg_video_classification"
classification = (spark.table(classification_table) if spark.catalog.tableExists(classification_table)
                  else spark.createDataFrame([], "video_id STRING, video_type STRING, confidence DOUBLE, is_food_review BOOLEAN, classified_at TIMESTAMP"))

categories = spark.createDataFrame([
    ("1", "Film & Animation"), ("2", "Autos & Vehicles"), ("10", "Music"), ("15", "Pets & Animals"),
    ("17", "Sports"), ("19", "Travel & Events"), ("20", "Gaming"), ("22", "People & Blogs"),
    ("23", "Comedy"), ("24", "Entertainment"), ("25", "News & Politics"), ("26", "Howto & Style"),
    ("27", "Education"), ("28", "Science & Technology"), ("29", "Nonprofits & Activism"),
], "category_id STRING, category_name STRING")

## Incremental: videos read **or** classified in the window

A video can get its `video_type` without a new read, so both sources give changed keys.

In [ ]:
if load_type == "incremental":
    read_in_window = (stg_videos
                      .filter((F.col("snapshot_date") >= from_date) & (F.col("snapshot_date") < to_date))
                      .select("video_id"))
    classified_day = F.to_date(F.from_utc_timestamp("classified_at", TZ))
    classified_in_window = (classification
                            .filter((classified_day >= from_date) & (classified_day < to_date))
                            .select("video_id"))
    changed = read_in_window.union(classified_in_window).distinct()
    stg_videos_latest = stg_videos_latest.join(changed, "video_id", "left_semi")

# Transformation

`duration_seconds` from the ISO 8601 duration (`PT1H2M3S` → 3723).

In [ ]:
def duration_part(unit):
    x = F.regexp_extract(F.col("video.duration"), rf"(\d+){unit}", 1)   # '' when the unit is missing
    return F.when(x != "", x.cast("int")).otherwise(F.lit(0))

result_df = (
    stg_videos_latest.alias("video")
    .join(categories.alias("category"), on=F.col("video.category_id") == F.col("category.category_id"), how="left")
    .join(classification.alias("class"), on=F.col("video.video_id") == F.col("class.video_id"), how="left")
    .select(
        F.xxhash64("video.video_id").alias("video_key"),
        F.col("video.video_id").cast("string").alias("video_id"),
        F.col("video.title").cast("string").alias("title"),
        F.col("video.description").cast("string").alias("description"),
        F.col("video.tags").alias("tags"),
        F.col("video.category_id").cast("string").alias("category_id"),
        F.col("category.category_name").cast("string").alias("category_name"),
        F.col("video.duration").cast("string").alias("duration_iso"),
        F.when(F.col("video.duration").isNotNull(),
               3600 * duration_part("H") + 60 * duration_part("M") + duration_part("S")).alias("duration_seconds"),
        F.col("video.definition").cast("string").alias("definition"),
        F.col("video.has_caption").cast("boolean").alias("has_caption"),
        F.col("video.thumbnail_url").cast("string").alias("thumbnail_url"),
        F.col("video.video_url").cast("string").alias("video_url"),
        F.col("video.channel_id").cast("string").alias("channel_id"),
        F.col("video.published_at").cast("timestamp").alias("published_at"),
        F.date_format(F.from_utc_timestamp("video.published_at", TZ), "yyyyMMdd").cast("int").alias("published_date_key"),
        F.col("class.video_type").cast("string").alias("video_type"),
        F.col("class.confidence").cast("double").alias("video_type_confidence"),
        F.coalesce(F.col("class.is_food_review"), F.lit(True)).alias("is_food_review"),   # not classified yet = true
        F.col("video.created_at").cast("timestamp").alias("created_at"),
        F.col("video.snapshot_ts").cast("timestamp").alias("modified_at"),
    )
)

## Add metadata

In [ ]:
insert_date = datetime.now()
etl_by = "Databricks"
result_df = result_df.withColumn("etl_date", F.lit(insert_date)).withColumn("etl_by", F.lit(etl_by))

## Add checksums

In [ ]:
primary_key = ["video_id"]
result_df = add_checksums(result_df, primary_key, exclude=["video_key", "created_at", "modified_at"])

## Create table

In [ ]:
path = f"{gold}.dim_video_scd1"
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {path} (
        video_key BIGINT, video_id STRING, title STRING, description STRING, tags ARRAY<STRING>, category_id STRING,
        category_name STRING, duration_iso STRING, duration_seconds INT, definition STRING, has_caption BOOLEAN,
        thumbnail_url STRING, video_url STRING, channel_id STRING, published_at TIMESTAMP, published_date_key INT,
        video_type STRING, video_type_confidence DOUBLE, is_food_review BOOLEAN,
        created_at TIMESTAMP, modified_at TIMESTAMP, etl_date TIMESTAMP, etl_by STRING,
        primary_checksum STRING, record_checksum STRING
    ) USING DELTA
    COMMENT 'Video dimension, SCD 1 (poc checksum method). video_key = xxhash64(video_id), -1 = Unknown.'
""")
result_df = result_df.select(*spark.table(path).columns)

# Write to Table

**full**: overwrite (poc `full_refresh`). **incremental**: MERGE on `primary_checksum`; update only when `record_checksum` changed and the source row is not older (`s.modified_at >= t.modified_at`, so a backfill of an old day never overwrites newer values); insert new keys.

In [ ]:
if load_type == "full":
    full_refresh(result_df, path)
else:
    result_df.createOrReplaceTempView("source")
    spark.sql(f"""
        MERGE INTO {path} AS t
        USING source AS s
        ON t.primary_checksum = s.primary_checksum
        WHEN MATCHED AND t.record_checksum <> s.record_checksum
             AND (t.modified_at IS NULL OR s.modified_at >= t.modified_at) THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

## Unknown row

Key −1, natural key NULL: facts with no match point to it. Inserted only when missing (no empty commit).

In [ ]:
if spark.table(path).where("video_key = -1").isEmpty():
    spark.sql(f"INSERT INTO {path} (video_key, title, category_name, etl_date, etl_by) VALUES (-1, 'Unknown', 'Unknown', current_timestamp(), 'Databricks')")

## Check

In [ ]:
dup = spark.sql(f"SELECT COUNT(*) - COUNT(DISTINCT video_id) FROM {path} WHERE video_id IS NOT NULL").first()[0]
assert dup == 0, f"{path}: {dup} duplicate video_id"
last = spark.sql(f"DESCRIBE HISTORY {path} LIMIT 2").where("operation <> 'WRITE'").select("operation", "operationMetrics").first()
print(f"✅ {path}: {spark.table(path).count()} rows", "·", last["operation"], dict(last["operationMetrics"] or {}) if last else "")